# 🤖 Selenium 입문

---

> **Selenium** 은 파이썬 코드로 실제 웹 브라우저를 직접 조작하는 자동화 도구입니다.  
> JavaScript로 동적으로 렌더링되는 페이지, 로그인이 필요한 페이지, 버튼 클릭 후 나타나는 데이터 등  
> `requests + BeautifulSoup` 만으로는 수집할 수 없는 상황에서 사용합니다.

## 📋 전체 학습 흐름

```
Part 1. 환경 설정
  1-1. 라이브러리 설치 및 import
  1-2. 드라이버 옵션 설정
  1-3. 브라우저 열기 & 페이지 이동

Part 2. 요소 탐색
  2-1. By 선택자 종류 전체 정리
  2-2. find_element vs find_elements
  2-3. CSS Selector vs XPath 비교

Part 3. 대기 전략
  3-1. 암묵적 대기 (implicitly_wait)
  3-2. 명시적 대기 (WebDriverWait + EC)
  3-3. 대기 전략 비교 & 선택 기준

Part 4. 브라우저 인터랙션
  4-1. 클릭 & 텍스트 입력
  4-2. 스크롤 제어
  4-3. JavaScript 실행
  4-4. 요소 상태 확인

Part 5. 반복 자동화 패턴
  5-1. 더보기 버튼 반복 클릭
  5-2. 무한 스크롤 처리

Part 6. Selenium + BS4 하이브리드 수집
  6-1. page_source 활용
  6-2. 네이버 뉴스 실전 수집

Part 7. 자주 하는 실수 & 팁
```

### ⚠️ Selenium 사용 전 주의사항

| 항목           | 내용                                                               |
| :------------- | :----------------------------------------------------------------- |
| **속도**       | requests보다 느림 — 정적 페이지는 requests+BS4 사용 권장           |
| **대기**       | 클릭 후 즉시 다음 코드 실행 시 오류 — 반드시 대기 로직 필요        |
| **자원**       | 브라우저 프로세스가 백그라운드 누적 — 작업 후 `driver.quit()` 필수 |
| **봇 탐지**    | 자동화 탐지 우회 옵션 설정 권장                                    |
| **robots.txt** | 크롤링 허용 여부 반드시 확인                                       |


In [1]:
# 필요한 라이브러리를 불러옵니다.
import re
import time
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Selenium 코어
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys                   # 키보드 입력
from selenium.webdriver.common.action_chains import ActionChains  # 마우스 동작
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import (
    NoSuchElementException,
    TimeoutException,
    StaleElementReferenceException,
)

SITE_URL = 'https://crawl-da.netlify.app/selenium_crawl'
print('라이브러리 로드 완료!')


라이브러리 로드 완료!


---

# 🔧 Part 1. 환경 설정


## 1-1. 드라이버 옵션 설정

`Options()` 객체로 브라우저 실행 전 환경을 설정합니다.

| 옵션                                            | 설명                                     |
| :---------------------------------------------- | :--------------------------------------- |
| `window-size=너비,높이`                         | 창 크기 고정 (반응형 레이아웃 변화 방지) |
| `--headless`                                    | 브라우저 창 없이 백그라운드 실행         |
| `--no-sandbox`                                  | 리눅스/서버 환경에서 필수                |
| `--disable-dev-shm-usage`                       | 메모리 부족 방지                         |
| `--disable-blink-features=AutomationControlled` | 자동화 탐지 우회                         |
| `user-agent=...`                                | 브라우저처럼 보이도록 헤더 위장          |
| `--disable-gpu`                                 | headless 모드에서 GPU 오류 방지          |

> 💡 로컬 환경에서 시각적으로 확인할 때는 `--headless` 를 빼고 실행하세요.


In [2]:
def make_driver(headless: bool = False) -> webdriver.Chrome:
    """용도에 맞는 Chrome 드라이버를 생성하여 반환합니다."""
    options = Options()

    # 창 크기 고정 (반응형 레이아웃 변화 방지)
    options.add_argument('window-size=1400,900')
    options.add_argument('lang=ko-KR')

    # 봇 탐지 우회 & 안정화 옵션
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )

    if headless:
        options.add_argument('--headless')
        options.add_argument('--disable-gpu')

    driver = webdriver.Chrome(options=options)
    return driver

print('make_driver 함수 정의 완료')
print('headless=False  → 브라우저 창이 열림 (시각적 확인 가능)')
print('headless=True   → 백그라운드 실행 (서버·배포 환경에서 사용)')


make_driver 함수 정의 완료
headless=False  → 브라우저 창이 열림 (시각적 확인 가능)
headless=True   → 백그라운드 실행 (서버·배포 환경에서 사용)


## 1-2. 브라우저 열기 & 페이지 이동

| 메서드                               | 설명                           |
| :----------------------------------- | :----------------------------- |
| `driver.get(url)`                    | 지정 URL로 이동                |
| `driver.title`                       | 현재 페이지 `<title>` 텍스트   |
| `driver.current_url`                 | 현재 페이지 URL                |
| `driver.page_source`                 | 현재 렌더링된 전체 HTML 문자열 |
| `driver.back()` / `driver.forward()` | 뒤로/앞으로                    |
| `driver.refresh()`                   | 새로고침                       |
| `driver.quit()`                      | 드라이버 종료 (반드시 호출)    |
| `driver.close()`                     | 현재 탭만 닫기                 |


In [3]:
# 브라우저를 열고 연습 페이지에 접속합니다.
driver = make_driver(headless=False)   # headless=True 로 바꾸면 창 없이 실행
driver.get(SITE_URL)

# 대기 객체 생성 (최대 5초 대기)
wait = WebDriverWait(driver, 5)

print('페이지 제목:   ', driver.title)
print('현재 URL:      ', driver.current_url)
print('HTML 길이:     ', len(driver.page_source), '자')


페이지 제목:    웹 크롤링 실습장 - 동적 데이터(Selenium)
현재 URL:       https://crawl-da.netlify.app/selenium_crawl
HTML 길이:      4659 자


In [4]:
# 브라우저 프록세스 종료
driver.quit()

### 🧩 확인 문제 1

드라이버를 생성하고 `https://crawl-da.netlify.app` 에 접속한 뒤,  
페이지 제목과 현재 URL을 출력하세요.


In [5]:
# ✅ 정답
test_driver = make_driver(headless=True)  # 창 없이 실행
test_driver.get('https://crawl-da.netlify.app')

print('제목:', test_driver.title)
print('URL: ', test_driver.current_url)

test_driver.quit()  # 종료

# 💡 해설:
# driver.get(url) → 페이지 이동
# driver.title    → <title> 태그 텍스트
# driver.quit()   → 브라우저 프로세스 종료 (메모리 누수 방지)


제목: [모두연] 데이터 분석 과정 - BS4 연습
URL:  https://crawl-da.netlify.app/


---

# 🔍 Part 2. 요소 탐색


## 2-1. By 선택자 종류 전체 정리

`find_element()` / `find_elements()` 에서 요소를 찾는 기준을 `By` 클래스로 지정합니다.

| By 상수                | 설명                        | 예시                              |
| :--------------------- | :-------------------------- | :-------------------------------- |
| `By.ID`                | id 속성                     | `By.ID, 'main-title'`             |
| `By.CLASS_NAME`        | class 속성 (단일 클래스)    | `By.CLASS_NAME, 'article-item'`   |
| `By.TAG_NAME`          | HTML 태그명                 | `By.TAG_NAME, 'a'`                |
| `By.NAME`              | name 속성 (주로 폼 입력)    | `By.NAME, 'query'`                |
| `By.CSS_SELECTOR`      | CSS 선택자 전체 문법        | `By.CSS_SELECTOR, '#wrap .title'` |
| `By.XPATH`             | XPath 표현식                | `By.XPATH, '//*[@id="title"]'`    |
| `By.LINK_TEXT`         | `<a>` 태그의 정확한 텍스트  | `By.LINK_TEXT, '더보기'`          |
| `By.PARTIAL_LINK_TEXT` | `<a>` 태그 텍스트 부분 일치 | `By.PARTIAL_LINK_TEXT, '더보'`    |

> 💡 **추천 순서**: `By.CSS_SELECTOR` → `By.ID` → `By.XPATH`  
> CSS Selector 가 간결하고 빠릅니다. 텍스트로 찾거나 부모 방향 탐색이 필요할 때만 XPath를 씁니다.


In [6]:
# 브라우저를 열고 연습 페이지에 접속합니다.
driver = make_driver(headless=False)   # headless=True 로 바꾸면 창 없이 실행
driver.get(SITE_URL)

# 대기 객체 생성 (최대 5초 대기)
wait = WebDriverWait(driver, 5)

In [7]:
# By 선택자 종류별 비교 실습
print('=== By.ID ===')
el = driver.find_element(By.ID, 'main-title')
print(el.text)

print()
print('=== By.CSS_SELECTOR (동일 요소) ===')
el2 = driver.find_element(By.CSS_SELECTOR, '#main-title')
print(el2.text)

print()
print('=== By.XPATH (동일 요소) ===')
el3 = driver.find_element(By.XPATH, '//*[@id="main-title"]')
print(el3.text)

print()
print('=== By.CLASS_NAME ===')
# ⚠️ CLASS_NAME 은 단일 클래스명만 가능 — 여러 클래스면 CSS_SELECTOR 사용
items = driver.find_elements(By.CLASS_NAME, 'article-item')
print(f'article-item 개수: {len(items)}개')

print()
print('=== By.TAG_NAME ===')
links = driver.find_elements(By.TAG_NAME, 'a')
print(f'전체 <a> 태그 수: {len(links)}개')


=== By.ID ===
Selenium 동적 크롤링 연습

=== By.CSS_SELECTOR (동일 요소) ===
Selenium 동적 크롤링 연습

=== By.XPATH (동일 요소) ===
Selenium 동적 크롤링 연습

=== By.CLASS_NAME ===
article-item 개수: 2개

=== By.TAG_NAME ===
전체 <a> 태그 수: 0개


## 2-2. find_element vs find_elements

| 메서드          |       반환        | 탐색 실패 시                  | 사용 목적                 |
| :-------------- | :---------------: | :---------------------------- | :------------------------ |
| `find_element`  |  WebElement 1개   | `NoSuchElementException` 발생 | 페이지에 하나만 있는 요소 |
| `find_elements` | 리스트 (0개 이상) | 빈 리스트 `[]` 반환           | 목록, 반복 요소           |

```python
# ❌ 없는 요소에 find_element → 즉시 오류
driver.find_element(By.ID, 'nonexistent')  # NoSuchElementException!

# ✅ 안전하게 확인하는 방법
elements = driver.find_elements(By.ID, 'nonexistent')
if elements:
    print(elements[0].text)
else:
    print('요소 없음')
```


In [8]:
print('=== find_element: 단일 요소 ===')
title = driver.find_element(By.CSS_SELECTOR, '#main-title')
print('제목:', title.text)

print()
print('=== find_elements: 복수 요소 ===')
cards = driver.find_elements(By.CSS_SELECTOR, 'li.article-item')
print(f'기사 카드: {len(cards)}개')

for idx, card in enumerate(cards, 1):
    # 카드 객체 안에서 다시 find_element 사용 (범위 좁힘)
    title_el  = card.find_element(By.CLASS_NAME, 'article-title')
    meta_el   = card.find_element(By.CLASS_NAME, 'meta-info')
    print(f'  {idx}. {title_el.text} | {meta_el.text}')

print()
print('=== 없는 요소 안전 처리 ===')
els = driver.find_elements(By.ID, 'nonexistent-id')
print('없는 요소 결과:', els)  # [] 빈 리스트, 오류 없음


=== find_element: 단일 요소 ===
제목: Selenium 동적 크롤링 연습

=== find_elements: 복수 요소 ===
기사 카드: 2개
  1. 파이썬 웹 크롤링 핵심 가이드 | 조회수: 1,024 | 2026-03-09
  2. Pandas로 시작하는 데이터 전처리 | 조회수: 856 | 2026-03-08

=== 없는 요소 안전 처리 ===
없는 요소 결과: []


## 2-3. CSS Selector vs XPath 비교

| 비교 기준          | CSS Selector                      | XPath                                    |
| :----------------- | :-------------------------------- | :--------------------------------------- |
| **문법**           | `#id`, `.class`, `태그 > 자식`    | `//*[@id='main']`, `//div[@class='box']` |
| **속도**           | 빠름                              | 상대적으로 느림                          |
| **가독성**         | 간결                              | 길고 복잡                                |
| **텍스트 탐색**    | ❌ 불가                           | ✅ `//button[text()='더보기']`           |
| **부모 방향 탐색** | ❌ 불가                           | ✅ `//span/parent::div`                  |
| **n번째 요소**     | `:nth-child(n)` (lxml만) / 인덱싱 | `(//li)[2]`                              |

### XPath 핵심 문법

```
//태그                    : 문서 전체에서 해당 태그 탐색
//*                       : 모든 태그
//태그[@속성='값']         : 속성 조건
//태그[contains(@class,'값')] : class 부분 포함
//태그[text()='텍스트']   : 텍스트 일치
//부모/자식               : 직계 자식
//부모//자손              : 모든 자손
//태그/..                 : 부모 요소
(//태그)[n]               : n번째 요소 (1-based)
```


In [9]:
print('=== CSS Selector vs XPath — 동일 요소 탐색 ===')

# 1. id 로 찾기
css1   = driver.find_element(By.CSS_SELECTOR, '#main-title').text
xpath1 = driver.find_element(By.XPATH, '//*[@id="main-title"]').text
print(f'CSS:   {css1}')
print(f'XPath: {xpath1}')

print()

# 2. 클래스로 찾기
css2   = [el.text for el in driver.find_elements(By.CSS_SELECTOR, '.article-title')]
xpath2 = [el.text for el in driver.find_elements(By.XPATH, '//*[contains(@class,"article-title")]')]
print('CSS   article-title:', css2)
print('XPath article-title:', xpath2)

print()

# 3. XPath 고유 기능: 텍스트로 버튼 찾기
# CSS Selector 로는 텍스트 조건 불가
try:
    btn_xpath = driver.find_element(By.XPATH, '//button[contains(text(),"더보기")]')
    print('XPath 텍스트 탐색:', btn_xpath.text)
except NoSuchElementException:
    print('버튼 없음 — 페이지 구조 확인 필요')

print()

# 4. XPath: n번째 요소
first_card  = driver.find_element(By.XPATH, '(//li[contains(@class,"article-item")])[1]')
second_card = driver.find_element(By.XPATH, '(//li[contains(@class,"article-item")])[2]')
print('첫 번째 카드:', first_card.find_element(By.CLASS_NAME, 'article-title').text)
print('두 번째 카드:', second_card.find_element(By.CLASS_NAME, 'article-title').text)


=== CSS Selector vs XPath — 동일 요소 탐색 ===
CSS:   Selenium 동적 크롤링 연습
XPath: Selenium 동적 크롤링 연습

CSS   article-title: ['파이썬 웹 크롤링 핵심 가이드', 'Pandas로 시작하는 데이터 전처리']
XPath article-title: ['파이썬 웹 크롤링 핵심 가이드', 'Pandas로 시작하는 데이터 전처리']

XPath 텍스트 탐색: 기사 더보기 (Click!)

첫 번째 카드: 파이썬 웹 크롤링 핵심 가이드
두 번째 카드: Pandas로 시작하는 데이터 전처리


### 🧩 확인 문제 2

CSS Selector 와 XPath 두 가지 방법으로 현재 페이지의 모든 기사 제목을 리스트로 추출하세요.

```
예상 결과: 두 결과가 동일해야 함
```


In [10]:
# ✅ 정답
css_titles   = [el.text for el in driver.find_elements(By.CSS_SELECTOR, '.article-title')]
xpath_titles = [el.text for el in driver.find_elements(By.XPATH, '//*[contains(@class,"article-title")]')]

print('CSS Selector:', css_titles)
print('XPath:        ', xpath_titles)
print('결과 일치:', css_titles == xpath_titles)

# 💡 해설:
# CSS  .article-title  → class에 'article-title' 포함
# XPath contains(@class,'article-title') → 동일 조건
# 기본적으로 CSS Selector가 간결하므로 CSS Selector를 먼저 시도하세요


CSS Selector: ['파이썬 웹 크롤링 핵심 가이드', 'Pandas로 시작하는 데이터 전처리']
XPath:         ['파이썬 웹 크롤링 핵심 가이드', 'Pandas로 시작하는 데이터 전처리']
결과 일치: True


In [11]:
driver.quit()

---

# ⏱️ Part 3. 대기 전략

Selenium 에서 가장 중요한 개념 중 하나입니다.  
페이지가 완전히 로딩되기 전에 요소를 탐색하면 `NoSuchElementException` 이 발생합니다.


## 3-1. 암묵적 대기 (implicitly_wait)

드라이버 전체에 적용되는 **전역 대기 설정**입니다.  
요소를 찾을 때 지정한 시간 동안 자동으로 재시도합니다.

```python
driver.implicitly_wait(5)   # 최대 5초 대기 (한 번만 설정하면 전체 적용)
```

| 장점                   | 단점                                            |
| :--------------------- | :---------------------------------------------- |
| 코드 한 줄로 전체 적용 | 특정 조건(클릭 후 새 요소 등)에는 적합하지 않음 |
| 간단함                 | 정확한 로딩 시점을 감지하지 못함                |


In [12]:
# 암묵적 대기 설정 예시 (실제 적용은 드라이버 생성 직후 한 번만)
# driver.implicitly_wait(5)

# 암묵적 대기가 설정된 드라이버 예시
demo_driver = make_driver(headless=False)
demo_driver.implicitly_wait(5)   # 요소 탐색 시 최대 5초 대기
demo_driver.get(SITE_URL)

title = demo_driver.find_element(By.ID, 'main-title')
print('암묵적 대기 후 제목:', title.text)

demo_driver.quit()


암묵적 대기 후 제목: Selenium 동적 크롤링 연습


## 3-2. 명시적 대기 (WebDriverWait + EC)

특정 **조건이 충족될 때까지** 대기합니다. 더 정밀하고 안정적입니다.

```python
wait = WebDriverWait(driver, 10)   # 최대 10초 대기
element = wait.until(EC.presence_of_element_located((By.ID, 'title')))
```

### 자주 쓰는 EC (expected_conditions) 목록

| EC 함수                           | 조건 설명                              |
| :-------------------------------- | :------------------------------------- |
| `presence_of_element_located`     | DOM에 요소가 존재할 때                 |
| `visibility_of_element_located`   | 요소가 화면에 보일 때                  |
| `element_to_be_clickable`         | 요소가 클릭 가능한 상태일 때           |
| `text_to_be_present_in_element`   | 요소 내 특정 텍스트가 나타날 때        |
| `invisibility_of_element_located` | 요소가 사라질 때 (로딩 스피너 제거 등) |
| `alert_is_present`                | 알림창이 나타날 때                     |
| `lambda d: 조건식`                | 커스텀 조건 (가장 유연함)              |


In [13]:
driver = make_driver(headless=False)
wait = WebDriverWait(driver, 5)
driver.get(SITE_URL)

print('=== EC.presence_of_element_located ===')
# DOM에 요소가 존재할 때까지 대기
el = wait.until(EC.presence_of_element_located((By.ID, 'main-title')))
print('요소 발견:', el.text)

print()
print('=== EC.visibility_of_element_located ===')
# 요소가 화면에 보일 때까지 대기
el2 = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, 'li.article-item')))
print('보이는 요소 발견:', el2.find_element(By.CLASS_NAME, 'article-title').text)

print()
print('=== EC.element_to_be_clickable ===')
# 클릭 가능한 상태일 때까지 대기 (버튼 클릭 전에 사용)
btn = wait.until(EC.element_to_be_clickable((By.ID, 'load-more-btn')))
print('클릭 가능한 버튼 텍스트:', btn.text)

print()
print('=== lambda 커스텀 조건 ===')
# 기사가 2개 이상 로딩될 때까지 대기
wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, 'li.article-item')) >= 2)
count = len(driver.find_elements(By.CSS_SELECTOR, 'li.article-item'))
print(f'기사 {count}개 로딩 확인')


=== EC.presence_of_element_located ===
요소 발견: Selenium 동적 크롤링 연습

=== EC.visibility_of_element_located ===
보이는 요소 발견: 파이썬 웹 크롤링 핵심 가이드

=== EC.element_to_be_clickable ===
클릭 가능한 버튼 텍스트: 기사 더보기 (Click!)

=== lambda 커스텀 조건 ===
기사 2개 로딩 확인


## 3-3. 대기 전략 비교 & 선택 기준

| 상황                        | 권장 대기 방법                                       |
| :-------------------------- | :--------------------------------------------------- |
| 단순한 정적 페이지 탐색     | `implicitly_wait`                                    |
| 버튼 클릭 후 새 요소 대기   | `WebDriverWait + EC.presence_of_element_located`     |
| 클릭 가능 여부 확인 후 클릭 | `WebDriverWait + EC.element_to_be_clickable`         |
| 요소 개수 변화 감지         | `WebDriverWait + lambda`                             |
| 로딩 스피너 사라질 때까지   | `WebDriverWait + EC.invisibility_of_element_located` |

> ⚠️ `time.sleep()` 은 항상 지정 시간만큼 무조건 기다리므로 비효율적입니다.  
> 스크래핑을 위한 최소 딜레이에만 사용하고, 요소 대기에는 `WebDriverWait` 를 쓰세요.


### 🧩 확인 문제 3

`load-more-btn` 버튼이 클릭 가능한 상태가 될 때까지 대기한 뒤,  
버튼 텍스트를 출력하세요.

```
예상 결과: 버튼 텍스트 출력 (오류 없이)
```


In [14]:
# ✅ 정답
driver = make_driver(headless=False)
wait = WebDriverWait(driver, 5)
driver.get(SITE_URL)

btn = wait.until(EC.element_to_be_clickable((By.ID, 'load-more-btn')))
print('버튼 텍스트:', btn.text)

# 💡 해설:
# element_to_be_clickable → 요소가 존재하고 활성화된 상태일 때 반환
# 버튼이 disabled 상태이면 계속 대기 → TimeoutException 발생
# 클릭 전에 항상 이 대기를 먼저 하는 습관을 들이세요


버튼 텍스트: 기사 더보기 (Click!)


---

# 🖱️ Part 4. 브라우저 인터랙션


## 4-1. 클릭 & 텍스트 입력

| 메서드                                 | 설명               |
| :------------------------------------- | :----------------- |
| `element.click()`                      | 요소 클릭          |
| `element.send_keys('텍스트')`          | 텍스트 입력        |
| `element.send_keys(Keys.ENTER)`        | 엔터키 입력        |
| `element.send_keys(Keys.CONTROL, 'a')` | Ctrl+A (전체 선택) |
| `element.clear()`                      | 입력창 내용 지우기 |
| `element.get_attribute('value')`       | 입력된 값 확인     |


In [15]:
# ── 버튼 클릭 ────────────────────────────────────────────────────
print('=== 버튼 클릭 ===')

before_count = len(driver.find_elements(By.CSS_SELECTOR, 'li.article-item'))
print(f'클릭 전 기사 수: {before_count}개')

# 클릭 가능 상태 확인 후 클릭
btn = wait.until(EC.element_to_be_clickable((By.ID, 'load-more-btn')))
btn.click()

# 기사가 추가될 때까지 대기
wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, 'li.article-item')) > before_count)

after_count = len(driver.find_elements(By.CSS_SELECTOR, 'li.article-item'))
print(f'클릭 후 기사 수: {after_count}개')

print()


=== 버튼 클릭 ===
클릭 전 기사 수: 2개
클릭 후 기사 수: 3개



## 4-2. 스크롤 제어

스크롤은 JavaScript를 실행하거나 `Keys.PAGE_DOWN` 으로 처리합니다.

```python
# 페이지 맨 아래로 스크롤
driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')

# 특정 위치로 스크롤
driver.execute_script('window.scrollTo(0, 500)')

# 특정 요소가 보이는 위치로 스크롤
driver.execute_script('arguments[0].scrollIntoView()', element)
```


In [16]:
print('=== 스크롤 제어 ===')

# 현재 스크롤 위치 확인
scroll_y = driver.execute_script('return window.pageYOffset')
page_height = driver.execute_script('return document.body.scrollHeight')
print(f'현재 스크롤 위치: {scroll_y}px')
print(f'전체 페이지 높이: {page_height}px')

# 페이지 맨 아래로 스크롤
driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
time.sleep(0.5)  # 스크롤 후 렌더링 대기

scroll_y_after = driver.execute_script('return window.pageYOffset')
print(f'스크롤 후 위치: {scroll_y_after}px')

# 다시 맨 위로
driver.execute_script('window.scrollTo(0, 0)')
print('맨 위로 복귀 완료')

print()

# 특정 요소가 보이는 위치로 스크롤
last_card = driver.find_elements(By.CSS_SELECTOR, 'li.article-item')[-1]
driver.execute_script('arguments[0].scrollIntoView()', last_card)
print('마지막 카드로 스크롤 완료')


=== 스크롤 제어 ===
현재 스크롤 위치: 0px
전체 페이지 높이: 545px
스크롤 후 위치: 0px
맨 위로 복귀 완료

마지막 카드로 스크롤 완료


## 4-3. JavaScript 실행

`driver.execute_script()` 로 브라우저에서 직접 JavaScript를 실행할 수 있습니다.  
일반 Selenium 메서드로 클릭이 안 되거나, 숨겨진 요소를 다룰 때 유용합니다.

```python
# 기본 형태
driver.execute_script('JS 코드')

# arguments[0] 으로 Selenium 요소 전달
driver.execute_script('arguments[0].click()', element)

# 반환값 받기
result = driver.execute_script('return document.title')
```


In [17]:
print('=== JavaScript 실행 ===')

# 페이지 제목 가져오기
js_title = driver.execute_script('return document.title')
print('JS로 가져온 제목:', js_title)

# 현재 URL 가져오기
js_url = driver.execute_script('return window.location.href')
print('JS로 가져온 URL:', js_url)

print()

# JS로 클릭 (일반 .click()이 동작 안 할 때 대안)
btn = driver.find_element(By.ID, 'load-more-btn')
if btn.is_enabled():
    before = len(driver.find_elements(By.CSS_SELECTOR, 'li.article-item'))
    driver.execute_script('arguments[0].click()', btn)
    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, 'li.article-item')) > before)
    after = len(driver.find_elements(By.CSS_SELECTOR, 'li.article-item'))
    print(f'JS 클릭 성공: {before}개 → {after}개')
else:
    print('버튼 비활성화 상태')


=== JavaScript 실행 ===
JS로 가져온 제목: 웹 크롤링 실습장 - 동적 데이터(Selenium)
JS로 가져온 URL: https://crawl-da.netlify.app/selenium_crawl

JS 클릭 성공: 3개 → 4개


## 4-4. 요소 상태 확인

| 메서드/속성                     | 반환 | 설명                                 |
| :------------------------------ | :--: | :----------------------------------- |
| `element.is_displayed()`        | bool | 화면에 보이는지 여부                 |
| `element.is_enabled()`          | bool | 활성화 상태인지 (버튼 disabled 확인) |
| `element.is_selected()`         | bool | 체크박스/라디오가 선택됐는지         |
| `element.text`                  | str  | 요소의 텍스트                        |
| `element.get_attribute('속성')` | str  | HTML 속성값                          |
| `element.tag_name`              | str  | 태그명 (`div`, `a` 등)               |
| `element.size`                  | dict | `{'width': ..., 'height': ...}`      |
| `element.location`              | dict | `{'x': ..., 'y': ...}` 화면 좌표     |


In [18]:
btn = driver.find_element(By.ID, 'load-more-btn')

print('=== 요소 상태 확인 ===')
print('is_displayed():', btn.is_displayed())  # 화면에 보이는지
print('is_enabled():  ', btn.is_enabled())    # 활성화 상태인지
print('text:          ', btn.text)
print('tag_name:      ', btn.tag_name)
print('size:          ', btn.size)

print()

# get_attribute 활용
links = driver.find_elements(By.TAG_NAME, 'a')
print('=== 링크 href 속성 추출 ===')
for link in links[:3]:
    href = link.get_attribute('href')
    text = link.text
    print(f'  텍스트={text!r:20} href={href}')


=== 요소 상태 확인 ===
is_displayed(): True
is_enabled():   True
text:           기사 더보기 (Click!)
tag_name:       button
size:           {'height': 48, 'width': 1278}

=== 링크 href 속성 추출 ===


### 🧩 확인 문제 4

현재 웹 페이지에서 제목과 정보를 추출해보세요.
**출력:**

```
기사 4개:
  제목 : 파이썬 웹 크롤링 핵심 가이드
  정보 : 조회수: 1,024 | 2026-03-09

  제목 : Pandas로 시작하는 데이터 전처리
  정보 : 조회수: 856 | 2026-03-08

  제목 : [New] Selenium을 활용한 버튼 자동 클릭 기법
  정보 : 조회수: 1,542 | 2026-03-07

  제목 : [New] BeautifulSoup vs Selenium: 언제 무엇을 쓸까?
  정보 : 조회수: 2,301 | 2026-03-06
```


In [19]:
# ✅ 정답
articles = driver.find_elements(By.CLASS_NAME, 'article-item')

print(f'기사 {len(articles)}개:')
for article in articles:
    title = article.find_element(By.CLASS_NAME, 'article-title').text
    meta  = article.find_element(By.CLASS_NAME, 'meta-info').text
    print(f'  제목 : {title}')
    print(f'  정보 : {meta}')
    print()

# 💡 해설:
# find_elements(By.CLASS_NAME, 'article-item') → 기사 블록 전체 목록 수집
# article.find_element(...)                    → 부모 요소 안에서만 탐색 (범위 한정)
# .text                                        → 태그 안의 텍스트만 추출


기사 4개:
  제목 : 파이썬 웹 크롤링 핵심 가이드
  정보 : 조회수: 1,024 | 2026-03-09

  제목 : Pandas로 시작하는 데이터 전처리
  정보 : 조회수: 856 | 2026-03-08

  제목 : [New] Selenium을 활용한 버튼 자동 클릭 기법
  정보 : 조회수: 1,542 | 2026-03-07

  제목 : [New] BeautifulSoup vs Selenium: 언제 무엇을 쓸까?
  정보 : 조회수: 2,301 | 2026-03-06



---

# 🔄 Part 5. 반복 자동화 패턴


## 5-1. 더보기 버튼 반복 클릭

### 안전한 반복 클릭의 핵심 원칙

1. **루프 진입 전 상태 확인** — 버튼이 활성화 상태인지 먼저 확인
2. **클릭 전 상태 기록** — 현재 요소 개수를 변수에 저장
3. **클릭 후 변화 대기** — 개수가 늘어날 때까지 대기
4. **매 루프마다 요소 재탐색** — DOM 변경 후 기존 참조가 무효화될 수 있음 (`StaleElementReferenceException`)
5. **최대 횟수 제한** — 무한 루프 방지


In [20]:
def click_until_disabled(driver, wait, button_selector: str,
                         item_selector: str, max_clicks: int = 10) -> int:
    """
    버튼이 비활성화될 때까지 안전하게 반복 클릭합니다.

    Parameters
    ----------
    driver          : Selenium WebDriver
    wait            : WebDriverWait 객체
    button_selector : 클릭할 버튼의 CSS Selector
    item_selector   : 개수를 감지할 요소의 CSS Selector
    max_clicks      : 최대 클릭 횟수 (무한루프 방지)

    Returns
    -------
    int : 총 클릭 횟수
    """
    clicks = 0

    while clicks < max_clicks:
        # 매 루프마다 버튼을 새로 찾음 (DOM 변경으로 인한 StaleElementReferenceException 방지)
        try:
            button = wait.until(EC.presence_of_element_located(
                (By.CSS_SELECTOR, button_selector)
            ))
        except TimeoutException:
            print('[종료] 버튼을 찾을 수 없습니다.')
            break

        # 버튼 비활성화 → 종료
        if not button.is_enabled():
            print('[종료] 버튼이 비활성화 상태입니다. 모든 데이터 로딩 완료.')
            break

        before_count = len(driver.find_elements(By.CSS_SELECTOR, item_selector))

        time.sleep(0.5)   # 크롤링 예절
        button.click()
        clicks += 1

        # 아이템 개수가 늘어날 때까지 대기
        try:
            wait.until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, item_selector)) > before_count
            )
        except TimeoutException:
            print(f'[경고] {clicks}회 클릭 후 새 아이템이 나타나지 않음')
            break

        current = len(driver.find_elements(By.CSS_SELECTOR, item_selector))
        print(f'  {clicks}회 클릭 → {current}개')

    return clicks

# 실행
driver = make_driver(headless=False)
wait = WebDriverWait(driver, 5)
driver.get(SITE_URL)

print('더보기 버튼 반복 클릭 시작...')
total_clicks = click_until_disabled(
    driver, wait,
    button_selector='#load-more-btn',
    item_selector='li.article-item',
    max_clicks=10
)
print(f'\n완료. 총 {total_clicks}회 클릭')
print(f'최종 기사 수: {len(driver.find_elements(By.CSS_SELECTOR, "li.article-item"))}개')


더보기 버튼 반복 클릭 시작...
  1회 클릭 → 3개
  2회 클릭 → 4개
  3회 클릭 → 5개
[종료] 버튼이 비활성화 상태입니다. 모든 데이터 로딩 완료.

완료. 총 3회 클릭
최종 기사 수: 5개


## 5-2. 무한 스크롤 처리

스크롤을 내릴 때마다 새 콘텐츠가 로딩되는 패턴 (Instagram, Twitter 등)입니다.

```
흐름:
1. 현재 페이지 높이 기록
2. 맨 아래로 스크롤
3. 로딩 대기
4. 새 높이 확인 → 변화 없으면 종료
```


In [21]:
def scroll_to_bottom(driver, pause: float = 1.5, max_scrolls: int = 20) -> int:
    """
    페이지 끝까지 스크롤하며 무한 스크롤 콘텐츠를 모두 로딩합니다.

    Parameters
    ----------
    driver      : Selenium WebDriver
    pause       : 스크롤 후 대기 시간(초)
    max_scrolls : 최대 스크롤 횟수

    Returns
    -------
    int : 총 스크롤 횟수
    """
    last_height = driver.execute_script('return document.body.scrollHeight')
    scrolls = 0

    while scrolls < max_scrolls:
        # 맨 아래로 스크롤
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
        scrolls += 1
        time.sleep(pause)   # 새 콘텐츠 로딩 대기

        new_height = driver.execute_script('return document.body.scrollHeight')

        if new_height == last_height:
            print(f'[종료] 더 이상 새 콘텐츠 없음. 총 {scrolls}회 스크롤')
            break

        print(f'  {scrolls}회 스크롤: {last_height}px → {new_height}px')
        last_height = new_height

    return scrolls

# 연습 사이트는 무한 스크롤이 아니므로 패턴 확인용 코드만 출력
print('무한 스크롤 함수 정의 완료')
print('실제 무한 스크롤 사이트에서 사용 예시:')
print('  scroll_to_bottom(driver, pause=2.0, max_scrolls=30)')


무한 스크롤 함수 정의 완료
실제 무한 스크롤 사이트에서 사용 예시:
  scroll_to_bottom(driver, pause=2.0, max_scrolls=30)


### 🧩 확인 문제 5

연습 사이트를 새로고침한 뒤, `click_until_disabled` 함수를 활용해  
모든 기사가 로딩된 후 기사 제목 목록을 출력하세요.


In [22]:
# ✅ 정답
driver.get(SITE_URL)
wait = WebDriverWait(driver, 5)

click_until_disabled(
    driver, wait,
    button_selector='#load-more-btn',
    item_selector='li.article-item',
    max_clicks=10
)

titles = [
    card.find_element(By.CLASS_NAME, 'article-title').text
    for card in driver.find_elements(By.CSS_SELECTOR, 'li.article-item')
]

print(f'\n전체 기사 제목 ({len(titles)}개):')
for i, t in enumerate(titles, 1):
    print(f'  {i}. {t}')

# 💡 해설:
# click_until_disabled → 버튼 비활성화까지 반복 클릭
# 이후 find_elements → 모든 카드를 한 번에 수집
# 반복문 안에서 find_element → 카드 안에서만 탐색 (범위 좁힘)


  1회 클릭 → 3개
  2회 클릭 → 4개
  3회 클릭 → 5개
[종료] 버튼이 비활성화 상태입니다. 모든 데이터 로딩 완료.

전체 기사 제목 (5개):
  1. 파이썬 웹 크롤링 핵심 가이드
  2. Pandas로 시작하는 데이터 전처리
  3. [New] Selenium을 활용한 버튼 자동 클릭 기법
  4. [New] BeautifulSoup vs Selenium: 언제 무엇을 쓸까?
  5. [New] 데이터 분석가를 위한 필수 라이브러리 Top 5


---

# 🔗 Part 6. Selenium + BS4 하이브리드 수집


## 6-1. page_source 활용 전략

Selenium으로 동적 콘텐츠를 모두 펼친 뒤, `page_source` 를 BS4로 파싱하는 패턴입니다.  
Selenium의 `find_elements` 보다 **BS4의 파싱이 훨씬 빠르고 유연**합니다.

```
Selenium 역할: 브라우저 조작 (클릭, 스크롤, 로그인 등)
BS4 역할:      완성된 HTML에서 데이터 추출 (빠른 파싱)
```

| 방법                   | 속도 | 유연성 | 추천 상황             |
| :--------------------- | :--: | :----: | :-------------------- |
| `driver.find_elements` | 느림 |  보통  | 실시간 요소 상태 확인 |
| `page_source` + BS4    | 빠름 |  높음  | 대량 데이터 추출      |


In [23]:
# 현재 렌더링된 HTML을 BS4로 파싱
page_html = driver.page_source
soup = BeautifulSoup(page_html, 'html.parser')

print('page_source 길이:', len(page_html), '자')
print()

# BS4로 기사 데이터 추출 (Selenium보다 빠름)
cards = soup.select('li.article-item')
print(f'BS4 파싱 결과: {len(cards)}개 카드')

rows = []
for card in cards:
    title = card.select_one('.article-title')
    meta  = card.select_one('.meta-info')
    link  = card.select_one('a')
    rows.append({
        '제목': title.get_text(strip=True) if title else None,
        '메타': meta.get_text(strip=True)  if meta  else None,
        '링크': link.get('href')            if link  else None,
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))


page_source 길이: 5459 자

BS4 파싱 결과: 5개 카드
                                         제목                      메타   링크
                           파이썬 웹 크롤링 핵심 가이드 조회수: 1,024 | 2026-03-09 None
                       Pandas로 시작하는 데이터 전처리   조회수: 856 | 2026-03-08 None
            [New] Selenium을 활용한 버튼 자동 클릭 기법 조회수: 1,542 | 2026-03-07 None
[New] BeautifulSoup vs Selenium: 언제 무엇을 쓸까? 조회수: 2,301 | 2026-03-06 None
           [New] 데이터 분석가를 위한 필수 라이브러리 Top 5 조회수: 3,110 | 2026-03-05 None


## 6-2. 네이버 뉴스 실전 수집

### 수집 파이프라인

|        단계        |   담당   | 설명                                  |
| :----------------: | :------: | :------------------------------------ |
| ① 목록 페이지 로딩 | Selenium | 더보기 버튼 클릭으로 기사 목록 펼치기 |
|    ② 링크 추출     |   BS4    | `page_source` 파싱으로 기사 URL 수집  |
| ③ 상세 페이지 수집 | requests | 빠른 HTTP 요청으로 상세 HTML 가져오기 |
|    ④ 본문 파싱     |   BS4    | 제목/요약/본문 추출                   |
|       ⑤ 저장       |  pandas  | DataFrame → CSV                       |


In [24]:
# ── 파싱 헬퍼 함수 (crawl.py 역할 내장) ──────────────────────────
def normalize_text(text: str) -> str:
    if not text: return ''
    return re.sub(r'\s+', ' ', text).strip()

def parse_naver_article(html: str, url: str = None, category: str = None) -> dict:
    """네이버 뉴스 기사 HTML 에서 제목/요약/본문을 추출합니다."""
    soup = BeautifulSoup(html, 'html.parser')

    # 제목
    title_el = soup.select_one('#title_area')
    title = normalize_text(title_el.get_text()) if title_el else None

    # 요약
    summary_el = soup.select_one('.media_end_summary')
    summary_list = [normalize_text(t) for t in summary_el.stripped_strings] if summary_el else []

    # 본문
    body_el = soup.select_one('#dic_area')
    body_lines = [normalize_text(t) for t in body_el.stripped_strings] if body_el else []

    # 본문 앞부분 요약 중복 제거
    if summary_list and body_lines[:len(summary_list)] == summary_list:
        body_lines = body_lines[len(summary_list):]

    result = {
        'title':   title,
        'summary': ' | '.join(summary_list) if summary_list else None,
        'content': normalize_text(' '.join(body_lines)),
        'content_len': len(' '.join(body_lines)),
    }
    if category: result['category'] = category
    if url:      result['url'] = url
    return result

print('파싱 함수 정의 완료')


파싱 함수 정의 완료


In [25]:
# ── Step 1: Selenium으로 목록 페이지 펼치기 ──────────────────────
options = Options()
options.add_argument('window-size=1400,900')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-blink-features=AutomationControlled')
options.add_argument(
    'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
)

news_driver = webdriver.Chrome(options=options)
news_wait   = WebDriverWait(news_driver, 10)

SECTION_URL = 'https://news.naver.com/breakingnews/section/103/241'
news_driver.get(SECTION_URL)

try:
    before_count = len(news_driver.find_elements(By.CSS_SELECTOR, 'li.sa_item'))
    news_driver.find_element(By.CSS_SELECTOR, 'div.section_more > a').click()
    news_wait.until(
        lambda d: len(d.find_elements(By.CSS_SELECTOR, 'li.sa_item')) > before_count
    )
    print('목록 더보기 클릭 성공')
except Exception as e:
    print('더보기 실패 (기존 목록으로 진행):', e)

# ── Step 2: BS4로 링크 추출 ──────────────────────────────────────
section_soup = BeautifulSoup(news_driver.page_source, 'html.parser')
link_els = section_soup.select('.section_latest_article .sa_item a.sa_thumb_link')
news_links = [el.get('href') for el in link_els]
print(f'수집된 링크: {len(news_links)}개')


목록 더보기 클릭 성공
수집된 링크: 70개


In [26]:
news_driver.quit()

In [27]:
# ── Step 3~4: requests + BS4 로 상세 기사 수집 ────────────────────
MAX_ARTICLES = 5
DELAY_SEC    = 1.0

detail_rows = []
print(f'상세 기사 수집 시작 (최대 {MAX_ARTICLES}건)...')

for i, url in enumerate(news_links[:MAX_ARTICLES], 1):
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = parse_naver_article(resp.text, url=url, category='건강/정보')
        detail_rows.append(data)
        print(f'  [{i}/{MAX_ARTICLES}] ✅ {data["title"]}')
    except Exception as e:
        print(f'  [{i}/{MAX_ARTICLES}] ❌ {url} | {e}')
        detail_rows.append({'url': url, 'title': None, 'summary': None,
                             'content': None, 'content_len': 0, 'error': str(e)})
    time.sleep(DELAY_SEC)

print(f'\n수집 완료: {len(detail_rows)}건')


상세 기사 수집 시작 (최대 5건)...
  [1/5] ✅ 당뇨병 걸리면 살 빠진다는데… 왜 나는 '비만 당뇨'지?
  [2/5] ✅ “2년간 못 걸었다” 하희라, 아픔 극복하고 아침마다 ‘이것’ 실천…왜?
  [3/5] ✅ "집 안 공기 점검해야"… 라돈, 난소암 위험 높인다
  [4/5] ✅ None
  [5/5] ✅ 피부과 가기 전 주방부터… ‘꿀 피부’ 만들어주는 식품 7가지

수집 완료: 5건


In [28]:
# ── Step 5: DataFrame 변환 & CSV 저장 ────────────────────────────
news_df = pd.DataFrame(detail_rows)

# 컬럼 순서 정리
col_order = ['title', 'summary', 'content', 'content_len', 'category', 'url']
col_order = [c for c in col_order if c in news_df.columns]
news_df = news_df[col_order]

output_path = 'naver_news_selenium.csv'
news_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {output_path}')
print()
news_df.head()


저장 완료: naver_news_selenium.csv



,title,summary,content,content_len,category,url
0,당뇨병 걸리면 살 빠진다는데… 왜 나는 '비만 당뇨'지?,NaN,체중이 줄지 않는다고 안심할 것이 아니라 비만한 상태 자체가 당뇨병의 초기 신호일 ...,1586,건강/정보,https://n.news.naver.com/mnews/article/346/000...
1,"“2년간 못 걸었다” 하희라, 아픔 극복하고 아침마다 ‘이것’ 실천…왜?",[셀럽헬스] 배우 하희라 아침 루틴,하희라는 20대 시절 촬영 중 오토바이에서 떨어져 등을 다쳤다. 이후 등부터 허리 ...,1391,건강/정보,https://n.news.naver.com/mnews/article/296/000...
2,"""집 안 공기 점검해야""… 라돈, 난소암 위험 높인다",NaN,눈에 보이지 않는 방사성 가스 '라돈'이 여성의 난소암 위험을 높일 수 있다는 연구...,1313,건강/정보,https://n.news.naver.com/mnews/article/346/000...
3,NaN,NaN,,0,건강/정보,https://n.news.naver.com/mnews/article/144/000...
4,피부과 가기 전 주방부터… ‘꿀 피부’ 만들어주는 식품 7가지,NaN,"영국 공인 영양사 멜리사 코헨은 ""일상 속 재료의 놀라운 효능을 활용하는 것이 건강...",1479,건강/정보,https://n.news.naver.com/mnews/article/346/000...


In [29]:
# ── 드라이버 종료 (반드시 실행) ──────────────────────────────────
for d in [driver, news_driver]:
    try:
        d.quit()
    except Exception:
        pass
print('모든 드라이버 종료 완료')


모든 드라이버 종료 완료


---

# ⚠️ Part 7. 자주 하는 실수 & 팁


## 7-1. 자주 하는 실수 모음


In [30]:
print('=' * 60)
print('실수 1. 대기 없이 클릭 후 바로 요소 탐색')
print('=' * 60)
print('''
# ❌ 클릭 직후 바로 탐색 → 아직 로딩 안 됐을 수 있음
driver.find_element(By.ID, 'load-more-btn').click()
cards = driver.find_elements(By.CSS_SELECTOR, 'li.article-item')  # 이전 개수 그대로!

# ✅ 클릭 후 변화 감지까지 대기
before = len(driver.find_elements(By.CSS_SELECTOR, 'li.article-item'))
driver.find_element(By.ID, 'load-more-btn').click()
wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, 'li.article-item')) > before)
cards = driver.find_elements(By.CSS_SELECTOR, 'li.article-item')  # 최신 목록
''')

print('=' * 60)
print('실수 2. StaleElementReferenceException')
print('=' * 60)
print('''
# ❌ DOM 변경 후 기존 요소 참조 사용 → StaleElementReferenceException!
btn = driver.find_element(By.ID, 'load-more-btn')
driver.refresh()      # 페이지 갱신으로 btn 참조 무효화
btn.click()           # 오류!

# ✅ 매 루프마다 요소를 새로 탐색
while True:
    btn = driver.find_element(By.ID, 'load-more-btn')  # 매번 새로 찾음
    if not btn.is_enabled(): break
    btn.click()
''')

print('=' * 60)
print('실수 3. driver.quit() 누락')
print('=' * 60)
print('''
# ❌ 작업 후 quit() 생략 → 백그라운드 크롬 프로세스 누적 → 메모리 낭비

# ✅ try/finally 패턴으로 반드시 종료 보장
driver = make_driver()
try:
    driver.get(url)
    # ... 작업 ...
finally:
    driver.quit()   # 오류가 나도 반드시 실행됨
''')

print('=' * 60)
print('실수 4. time.sleep() 남용')
print('=' * 60)
print('''
# ❌ 무조건 3초 대기 → 느리고 불안정
time.sleep(3)
cards = driver.find_elements(...)

# ✅ 조건 충족 즉시 진행 (최대 5초까지만 대기)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'li.article-item')))
cards = driver.find_elements(...)
''')

print('=' * 60)
print('실수 5. By.CLASS_NAME 에 여러 클래스 사용')
print('=' * 60)
print('''
# ❌ CLASS_NAME 은 단일 클래스명만 가능
driver.find_element(By.CLASS_NAME, 'article-item active')  # 오류!

# ✅ 복합 클래스는 CSS_SELECTOR 사용
driver.find_element(By.CSS_SELECTOR, '.article-item.active')
''')


실수 1. 대기 없이 클릭 후 바로 요소 탐색

# ❌ 클릭 직후 바로 탐색 → 아직 로딩 안 됐을 수 있음
driver.find_element(By.ID, 'load-more-btn').click()
cards = driver.find_elements(By.CSS_SELECTOR, 'li.article-item')  # 이전 개수 그대로!

# ✅ 클릭 후 변화 감지까지 대기
before = len(driver.find_elements(By.CSS_SELECTOR, 'li.article-item'))
driver.find_element(By.ID, 'load-more-btn').click()
wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, 'li.article-item')) > before)
cards = driver.find_elements(By.CSS_SELECTOR, 'li.article-item')  # 최신 목록

실수 2. StaleElementReferenceException

# ❌ DOM 변경 후 기존 요소 참조 사용 → StaleElementReferenceException!
btn = driver.find_element(By.ID, 'load-more-btn')
driver.refresh()      # 페이지 갱신으로 btn 참조 무효화
btn.click()           # 오류!

# ✅ 매 루프마다 요소를 새로 탐색
while True:
    btn = driver.find_element(By.ID, 'load-more-btn')  # 매번 새로 찾음
    if not btn.is_enabled(): break
    btn.click()

실수 3. driver.quit() 누락

# ❌ 작업 후 quit() 생략 → 백그라운드 크롬 프로세스 누적 → 메모리 낭비

# ✅ try/finally 패턴으로 반드시 종료 보장
driver = make_driver()
try:

## 7-2. Selenium vs requests 선택 기준

| 상황                    |        권장 도구        |
| :---------------------- | :---------------------: |
| 정적 HTML 페이지        |     requests + BS4      |
| JavaScript 렌더링 필요  |        Selenium         |
| 로그인 필요             |        Selenium         |
| 버튼 클릭 / 스크롤 필요 |        Selenium         |
| 대량 상세 페이지 수집   | requests (링크 확보 후) |
| 빠른 수집이 중요        |     requests + BS4      |

> 💡 **하이브리드 전략이 가장 효율적입니다:**  
> Selenium 으로 동적 목록을 펼친 뒤 → links 추출 → requests 로 상세 수집


## 7-3. 정리

```python
# 드라이버 생성 & 페이지 이동
driver = webdriver.Chrome(options=options)
driver.get(url)
driver.quit()   # 반드시 호출!

# 요소 탐색
driver.find_element(By.CSS_SELECTOR, '#id .class')   # 1개
driver.find_elements(By.CSS_SELECTOR, 'li.item')     # 여러 개

# 인터랙션
element.click()
element.send_keys('텍스트')
element.send_keys(Keys.ENTER)
element.clear()

# 상태 확인
element.is_enabled()          # 활성화 여부
element.is_displayed()        # 화면 표시 여부
element.get_attribute('href') # 속성값
element.text                  # 텍스트

# 대기
wait = WebDriverWait(driver, 10)
wait.until(EC.element_to_be_clickable((By.ID, 'btn')))
wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, 'li')) > 5)

# JS 실행
driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
driver.execute_script('arguments[0].click()', element)

# page_source → BS4
soup = BeautifulSoup(driver.page_source, 'html.parser')
```

---

> 🎉 수고하셨습니다! 실습 사이트: 👉 [crawl-da.netlify.app/selenium_crawl](https://crawl-da.netlify.app/selenium_crawl)
